# WSPS-GRPO no Kaggle (v38 + scaffold + juiz plumb + goldens)
Objetivo: GRPO com fases Ornith A+B+C. GPU0 = treino (policy v38, ~5GB);
GPU1 = juiz plumb-4b (~10GB). Juiz decide veto, nao gera texto.
Payload (dataset `lordnyx0/qwen3loop-grpo-wsps-data`, NOVO): `train_grpo_tool_v3.py`,
`rewards_tool_calling.py`, `scaffold_pool.py`, `grpo_wsps_v1.jsonl` (75),
`shell_golden_table.json` (352), `eval_probe_canonical_25.jsonl`, engine espelho.
Bundle (`lordnyx0/qwen3loop-tool-grpo-bundle`, ATUALIZAR p/ v38): `model/` v38mix-substituido
por v38_ep2 (safetensors+tokenizer+probe). Pesos do plumb baixados no kernel
(`hf download`, ~8GB, rev pinada) — nao vao no bundle.
Saidas em `/kaggle/working` (baixar ao fim): `grpo_wsps_v1/`, `wsps_rollouts.jsonl`,
`judge_verdicts_wsps.jsonl`, `plumb.log`, `data/tool_grpo_v3_training_log.json`.

## LANCAR (nao executado; comandos exatos, nesta ordem)
1. `kaggle datasets create -p Kaggle/kaggle_grpo_wsps` (dataset NOVO, 1a versao)
2. `kaggle datasets version -p Kaggle/kaggle_bundle -m "model v38_ep2 (era mix3)"`
3. Novo kernel a partir DESTE arquivo (slug sugerido `lordnyx0/qwen3loop-wsps-grpo`,
GPU T4 x2, internet ON): anexar `lordnyx0/qwen3loop-grpo-wsps-data` +
`lordnyx0/qwen3loop-tool-grpo-bundle`; `kaggle kernels push` APOS criar metadata
proprio (nao reutilizar o do kernel SFT).
4. Ao COMPLETE: baixar `grpo_wsps_v1.zip` + `wsps_rollouts.jsonl` + `judge_verdicts_wsps.jsonl`.
Estimativa: ~5h (150 steps two-stage) + download plumb ~8GB no inicio.
`kaggle_sft_data/` e kernel SFT intocados.


In [ ]:
# 1. GPUs Tesla T4 ativas
!nvidia-smi

In [ ]:
# 2. Dependencias (transformers recente p/ Qwen3_5 do juiz)
!pip uninstall -y torchao
!pip install -q -U transformers peft datasets accelerate huggingface_hub numpy

In [ ]:
# 3. Localizar datasets e montar /kaggle/working
import os, shutil

PAYLOAD, BUNDLE = None, None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'train_grpo_tool_v3.py' in files and 'grpo_wsps_v1.jsonl' in files:
        PAYLOAD = root
    if 'judge_v38_ep2_q8_0.gguf' in files and os.path.isfile(os.path.join(root, 'model', 'model.safetensors')):
        BUNDLE = root
print('PAYLOAD:', PAYLOAD)
print('BUNDLE:', BUNDLE)
assert PAYLOAD and BUNDLE, 'datasets incompletos (ver celula 0)'

W = '/kaggle/working'
for fn in ['train_grpo_tool_v3.py', 'rewards_tool_calling.py', 'scaffold_pool.py',
           'grpo_wsps_v1.jsonl', 'shell_golden_table.json', 'eval_probe_canonical_25.jsonl']:
    shutil.copy2(os.path.join(PAYLOAD, fn), os.path.join(W, fn))
shutil.copytree(os.path.join(PAYLOAD, 'qwen3loop_python'), os.path.join(W, 'engine', 'qwen3loop_python'), dirs_exist_ok=True)
MODEL_DIR = os.path.join(BUNDLE, 'model')
JUDGE_GGUF = os.path.join(BUNDLE, 'judge_v38_ep2_q8_0.gguf')
assert os.path.isfile(os.path.join(MODEL_DIR, 'latent_halting_probe.pt')), 'probe ausente no bundle'
print('payload + bundle OK; GGUF juiz:', round(os.path.getsize(JUDGE_GGUF) / 1e6), 'MB')
print('MODEL_DIR =', MODEL_DIR)


In [ ]:
# 4. Juiz plumb-4b na GPU2 (policy treina na GPU0). Pesos via download pinado.
import json as _json
import os as _os
import subprocess as _sp
import time as _time
import urllib.request as _url

_os.chdir('/kaggle/working')
REV_REPO = '0e6d558d3ea3002c300ce121db4d3e8df6ad792a'
REV_W = '55de037801a8a9b9de3db5c0e16cef86210c2186'
_sp.run(['curl', '-sL', '-o', 'plumb_server.py',
         f'https://raw.githubusercontent.com/crh225/plumb/{REV_REPO}/serve/plumb_server.py'],
        check=True)
assert _os.path.getsize('plumb_server.py') > 10000, 'plumb_server.py nao baixou'
# T4 (Turing, sm75) nao tem bf16: o servidor fixa torch.bfloat16 -> patch p/ fp16.
# Uma linha, deterministico; a calibracao T=2.07 tolera (threshold 0.8 robusto).
_src = open('plumb_server.py', encoding='utf-8').read()
assert 'dtype=torch.bfloat16' in _src, 'plumb_server.py mudou de formato!'
open('plumb_server.py', 'w', encoding='utf-8').write(
    _src.replace('dtype=torch.bfloat16', 'dtype=torch.float16'))
from huggingface_hub import snapshot_download
snapshot_download('crh225/plumb-4b', revision=REV_W, local_dir='plumb-4b',
                  local_dir_use_symlinks=False)
env = dict(_os.environ, CUDA_VISIBLE_DEVICES='1')
log = open('/kaggle/working/plumb.log', 'w')
_sp.Popen(['python', 'plumb_server.py', '--model', './plumb-4b', '--port', '8090',
           '--one-read', '--temperature', '2.07', '--noul-commit',
           '--score-temperature', '1.2'],
          stdout=log, stderr=_sp.STDOUT, stdin=_sp.DEVNULL, env=env, cwd='/kaggle/working')
ok = False
probe = _json.dumps({"state": "warm-up",
                     "questions": {"w": {"type": "noul", "instructions": "Is this warm?"}}}).encode()
for _ in range(120):
    _time.sleep(5)
    try:
        req = _url.Request('http://127.0.0.1:8090/v1/systemone', data=probe,
                           headers={'Content-Type': 'application/json'})
        with _url.urlopen(req, timeout=10) as r:
            d = _json.load(r)
            if 'noul' in d.get('answers', {}).get('w', {}):
                ok = True
                break
    except Exception:
        pass
assert ok, 'plumb nao subiu em 10min (ver plumb.log)'
print('juiz plumb UP :8090 na GPU2 (one-read T=2.07)')


In [ ]:
# 5. Exportar MODEL_DIR p/ a celula shell seguinte
import os
MODEL_DIR = [os.path.join(r, 'model') for r, d, f in os.walk('/kaggle/input') if 'model.safetensors' in f][0]
os.environ['MODEL_DIR'] = MODEL_DIR
print('MODEL_DIR =', MODEL_DIR)


In [ ]:
!cd /kaggle/working && python train_grpo_tool_v3.py --steps 2 --model_dir "$MODEL_DIR" --rollouts /kaggle/working/grpo_wsps_v1.jsonl --output_dir /kaggle/working/grpo_wsps_v1 --beta_kl 0.10 --len_w 0.10 --probe /kaggle/working/eval_probe_canonical_25.jsonl --eval_interval 25 --save_interval 25 --scaffold --scaffold_tokens 64 --judge_veto --judge_backend plumb --judge_url http://127.0.0.1:8090 --judge_threshold 0.8 --judge_log /kaggle/working/judge_verdicts_wsps.jsonl --shell_golden /kaggle/working/shell_golden_table.json --rollout_log /kaggle/working/wsps_rollouts.jsonl


In [ ]:
# 6. Empacotar saidas p/ download
!cd /kaggle/working && zip -r grpo_wsps_v1.zip grpo_wsps_v1/ wsps_rollouts.jsonl judge_verdicts_wsps.jsonl judge.log data/tool_grpo_v3_training_log.json
!ls -lh /kaggle/working/grpo_wsps_v1.zip
print('COMPLETE: baixar grpo_wsps_v1.zip + wsps_rollouts.jsonl + judge_verdicts_wsps.jsonl')
